# Three-month spatiotemporal follow-up — stronger XGBoost screen

This is the **larger follow-up** to the 650-home October shotgun: September–November 2025, all validation homes in each month, up to 40,000 historical training sales per monthly fold, and 500 trees per XGBoost training fit. Start in 2022 so the 730-day age filter can meaningfully differ from the legacy selection.

**Candidates, independently compared against the same baseline:**
- Baseline: 47-term Binary BoW XGBoost + original comparables.
- Feature changes: same-type quota, age-first, both.
- Residual corrections of the baseline: kNN, RBF Ridge, tensor-spline GAM approximation.

Residual models use **three earlier out-of-time calibration folds for each prediction month**; they do *not* learn residuals on the scored homes. These are still exploratory models: October 2025 was already used to choose the shortlist, and scores cannot be directly compared with the 7.82% 700-tree/full-history benchmark.

Each month has a separate folder and can be resumed after completion without redoing prior complete months. Partial or failed model runs are never silently included in pooled results. Your production `model_run.ipynb` and model defaults are untouched.

### Run after pulling main

```powershell
git pull origin main
uv sync
uv run python -m pytest tests/test_spacetime_three_month.py -q
uv run jupyter lab
```

You can also execute the same benchmark in a terminal:
```powershell
uv run python spacetime_three_month.py
```


In [ ]:
from IPython.display import display
from spacetime_three_month import run_three_month, MONTHS, FEATURES, RESIDUALS

TRAIN_START = "2022-01-01"
TRAIN_CAP = 40000
TREES = 500
VALIDATION_CAP = 1_000_000   # effectively: every home in each month
CALIBRATION_CAP = 1200
CALIBRATION_FOLDS = 3
BOOTSTRAP_REPEATS = 500
SEED = 42

FEATURE_VARIANTS = FEATURES
RESIDUAL_VARIANTS = RESIDUALS
OUTPUT_DIR = "outputs/spacetime_three_month"

print("Months:", MONTHS)
print("Feature variants:", FEATURE_VARIANTS)
print("Residual variants:", RESIDUAL_VARIANTS)
print("Sample cap:", TRAIN_CAP, "| trees:", TREES)

In [ ]:
pooled, monthly, by_type, predictions = run_three_month(
    months=MONTHS,
    train_start=TRAIN_START,
    train_cap=TRAIN_CAP,
    trees=TREES,
    validation_cap=VALIDATION_CAP,
    calibration_cap=CALIBRATION_CAP,
    calibration_folds=CALIBRATION_FOLDS,
    features=FEATURE_VARIANTS,
    residuals=RESIDUAL_VARIANTS,
    seed=SEED,
    bootstrap_repeats=BOOTSTRAP_REPEATS,
    output_dir=OUTPUT_DIR,
    resume=True,
)
display(pooled)
display(monthly)

In [ ]:
# Evaluate consistency across three chronological months.
columns = [
    "arm", "variant", "homes", "mdape", "mape",
    "delta_mdape_pp", "delta_mape_pp", "months_better_mdape",
    "homes_improved_fraction",
    "delta_mdape_ci_low", "delta_mdape_ci_high",
]
display(pooled[columns])

# Individual monthly deltas help guard against a one-month fluke.
monthly_delta = monthly.pivot_table(
    index=["arm", "variant"], columns="month",
    values="delta_mdape_pp",
)
display(monthly_delta.round(3))
print("Intervals pair-resample properties; they do not capture all market-month dependence.")

In [ ]:
# Property-type diagnostics; each group has at least 20 homes.
if not by_type.empty:
    display(by_type.sort_values(["house_type_name", "mdape"]).round(3))
else:
    print("No sufficiently large property-type groups in these rows.")

# Optional visual: bars show improvement relative to baseline.
import matplotlib.pyplot as plt
ranking = pooled.sort_values("delta_mdape_pp")
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(
    ranking["arm"] + " / " + ranking["variant"],
    -ranking["delta_mdape_pp"],
)
ax.axvline(0, color="gray", linewidth=1)
ax.set_xlabel("Reduction in MdAPE vs reduced-history baseline (percentage points)")
ax.set_title("Sep–Nov 2025 chronological comparison")
fig.tight_layout()
plt.show()

## How to interpret the second-stage results

- Judge both **pooled MdAPE and MAPE**, not just a single good subgroup.
- Prefer candidates with improvements over **multiple months**, not a very large win in one month.
- The bootstrap intervals describe paired uncertainty **conditional on these three months**, not uncertainty about a future housing market.
- **Age-first may now differ** because January 2022–2025 transactions are eligible for the training sample, including sales older than 730 days.
- Residual corrections are based on the **original comparable model**, not on the type-quota model. Combining them is a separate experiment requiring recalibration of the residual targets.
- After this, run the winning *feature change* through the full `model_run.ipynb` with 700 trees and the full historical training set. A winning *residual method* needs a deployment-ready chronological calibration pipeline before final confirmation.

Output files:
- `outputs/spacetime_three_month/pooled_summary.csv`
- `outputs/spacetime_three_month/monthly_summary.csv`
- `outputs/spacetime_three_month/house_type_summary.csv`
- `outputs/spacetime_three_month/predictions.csv`
- Three month-specific folders, supporting reuse/resume

The benchmark does **not** append speculative or unverified entries to `logs.md`.